# 01 — Scraping Wikipédia Arabe & Pipeline NLP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassinekamouss/fstt-dl-lab3-nlp-pytorch/blob/main/notebooks/01_scraping_and_dataset.ipynb)

**Université Abdelmalek Essaadi — FST Tanger — Master MBD — Deep Learning (Lab 3)**

Ce notebook réalise la collecte et la préparation complète du dataset pour la Partie 1 et la Partie 2 :
1. **Scraping Wikipédia Arabe (`BeautifulSoup4` + `requests`)** autour de la thématique **الذكاء الاصطناعي (Intelligence Artificielle)** sur 3 niveaux de pertinence (`core`, `related`, `off_topic`).
2. **Calcul du score de pertinence continu dans `[0, 10]`** au format exact `Text | Score`.
3. **Pipeline NLP Arabe complet** : nettoyage/normalisation (`PyArabic`), tokenization, filtrage des stop-words (`NLTK` + Wikipédia), comparaison **Stemming (`ISRI` / `Snowball`) vs Lemmatization (`Qalsadi`)**.
4. **Discrétisation** en 3 classes équilibrées (`Faible`, `Moyenne`, `Élevée`) + **EDA visuelle**.
5. **Split stratifié groupé par article (`title`)** anti-fuite de données (`data leakage`), construction du vocabulaire strictement sur `train`, et export pour les Notebooks `02` et `03`.

## 1. Bootstrap Google Colab / Drive & Environnement

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG") is not None
REPO_URL = "https://github.com/yassinekamouss/fstt-dl-lab3-nlp-pytorch.git"
REPO_NAME = "fstt-dl-lab3-nlp-pytorch"
USE_DRIVE = True  # Mettre False pour garder les donnees uniquement dans /content

if IN_COLAB:
    repo_dir = Path("/content") / REPO_NAME
    if not repo_dir.exists():
        !git clone {REPO_URL} {repo_dir}
    else:
        !git -C {repo_dir} pull --ff-only
    os.chdir(repo_dir)
else:
    cwd = Path.cwd().resolve()
    repo_dir = cwd.parent if cwd.name == "notebooks" else cwd
    os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print(f"Working directory: {Path.cwd()}")

In [ ]:
if IN_COLAB:
    !pip install -q beautifulsoup4 lxml requests pyyaml tqdm pandas numpy scikit-learn matplotlib seaborn nltk PyArabic qalsadi arabic-reshaper python-bidi

from src.colab_utils import load_config, print_system_info, seed_everything, setup_environment

paths = setup_environment(
    use_drive=USE_DRIVE,
    drive_dir="/content/drive/MyDrive/fstt-dl-lab3-nlp-pytorch",
    repo_root=Path.cwd(),
)
cfg = load_config(paths["config"])
seed_everything(cfg["project"]["seed"])
print_system_info()
print("Artifacts root:", paths["artifact_root"])

## 2. Scraping Multi-Niveaux sur Wikipédia Arabe (`BeautifulSoup4`)

Afin d'obtenir des textes couvrant toute la plage de pertinence `[0, 10]`, le scraper collecte des paragraphes depuis :
- **`core`** : articles centraux en Intelligence Artificielle (`الذكاء الاصطناعي`, `تعلم الآلة`, `التعلم العميق`, `معالجة اللغات الطبيعية`...).
- **`related`** : domaines informatiques/mathématiques connexes (`علم البيانات`, `خوارزمية`, `إحصاء`, `روبوتية`...).
- **`off_topic`** : thématiques sans lien (`تاريخ المغرب`, `كرة القدم`, `علم الفلك`, `الأدب العربي`...).

Le `Score` continu `[0, 10]` combine la similarité cosinus TF-IDF (leave-one-out) vis-à-vis du centroïde IA, la densité lexicale dans le lexique métier arabe, et un a priori borné lié au niveau de proximité Wikipédia.

In [ ]:
import pandas as pd
from src.scraper import ArabicWikipediaScraper

scraper = ArabicWikipediaScraper(
    output_dir=paths["data_raw"],
    config=cfg,
    cache_dir=paths["html_cache"],
)

raw_df = scraper.run(progress=True)
print(f"\nTotal paragraphes collectes : {len(raw_df)} | Articles distincts : {raw_df['title'].nunique()}")
print("\nRepartition par tier :")
print(raw_df["tier"].value_counts())
print("\nStatistiques des scores de pertinence :")
print(raw_df.groupby("tier")["score"].describe().round(2))

In [ ]:
# Apercu au format exact demande par le cahier des charges (Text | Score)
spec_df = pd.read_csv(paths["data_raw"] / "dataset_text_score.csv")
print(f"Fichier specification : {paths['data_raw'] / 'dataset_text_score.csv'} ({len(spec_df)} lignes)")

preview_rows = (
    raw_df.groupby("tier", as_index=False)
    .head(2)
    [["title", "tier", "score", "word_count", "text"]]
)
display(preview_rows)

## 3. Pipeline de Préprocessing NLP Arabe

Étapes appliquées à chaque paragraphe :
1. **Normalisation (`PyArabic`)** : suppression des URLs, balises, chiffres, diacritiques (*tashkeel*) et *tatweel*, unification des variantes d'Alef (`أ/إ/آ/ٱ → ا`) et Alef Maksura (`ى → ي`).
2. **Tokenization** : découpage propre en mots arabes (`araby.tokenize`).
3. **Stop-words** : liste `NLTK` arabe enrichie de termes récurrents Wikipédia (`انظر`, `ايضا`, `مراجع`, `حيث`, `وذلك`...), **normalisée** avec la même fonction pour éviter toute fuite lexicale.
4. **Stemming vs Lemmatization** : comparaison directe entre `ISRIStemmer`, `SnowballStemmer` et `Qalsadi Lemmatizer` (avec cache en mémoire et normalisation post-lemmatisation).

In [ ]:
from src.preprocessing import ArabicTextPreprocessor

preprocessor = ArabicTextPreprocessor()
processed_df, prep_stats = preprocessor.process_dataframe(
    raw_df,
    text_col="text",
    min_tokens=cfg["preprocessing"]["min_tokens_after_clean"],
    progress=True,
)

print("Resume du preprocessing :")
for k, v in prep_stats.items():
    print(f"  - {k}: {v}")

In [ ]:
import matplotlib.pyplot as plt

# Tableau comparatif avant/apres sur 3 exemples (incluant Snowball vs ISRI vs Qalsadi)
comparison_rows = []
for _, row in processed_df.head(3).iterrows():
    toks = str(row["text_nostop"]).split()[:12]
    isri = preprocessor.stem_tokens(toks)
    snow = preprocessor.snowball_stem_tokens(toks)
    lemma = str(row["text_lemmatized"]).split()[:12]
    comparison_rows.append(
        {
            "Original (extrait)": " ".join(str(row["text"]).split()[:12]),
            "Normalise + Sans Stopwords": " ".join(toks),
            "ISRI Stemmer": " ".join(isri),
            "Snowball Stemmer": " ".join(snow),
            "Qalsadi Lemmatizer": " ".join(lemma),
        }
    )
display(pd.DataFrame(comparison_rows))

# Visualisation de la reduction du vocabulaire
stages = ["Brut", "Normalise", "Sans Stopwords", "Lemmatise (Qalsadi)", "Stemme (ISRI)"]
vocab_sizes = [
    prep_stats["vocab_raw"],
    prep_stats["vocab_clean"],
    prep_stats["vocab_nostop"],
    prep_stats["vocab_lemmatized"],
    prep_stats["vocab_stemmed"],
]
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(stages, vocab_sizes, color=["#64748b", "#3b82f6", "#0ea5e9", "#10b981", "#f59e0b"])
for bar, val in zip(bars, vocab_sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 50, str(val), ha="center", va="bottom", fontweight="bold")
ax.set_title("Compression du Vocabulaire a travers le Pipeline NLP Arabe")
ax.set_ylabel("Tokens uniques")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(paths["figures"] / "vocab_reduction.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Discrétisation du Score & Analyse Exploratoire (EDA)

Transformation du `Score` continu `[0, 10]` en 3 classes pour l'évaluation des architectures de classification :
- **Classe 0** : `Faible (0-4)` — `[0.0, 4.0[`
- **Classe 1** : `Moyenne (4-7)` — `[4.0, 7.0[`
- **Classe 2** : `Élevée (7-10)` — `[7.0, 10.0]`

In [ ]:
from IPython.display import Image, display as ipy_display
from src.metrics import plot_eda_distributions
from src.preprocessing import discretize_scores

bins = cfg["preprocessing"]["discretization_bins"]
class_names = cfg["preprocessing"]["class_names"]

labels, label_names = discretize_scores(processed_df["score"], bins=bins, class_names=class_names)
processed_df["label"] = labels
processed_df["label_name"] = label_names

print("Distribution des classes :")
print(processed_df["label_name"].value_counts())
print("\nTableau croise Tier vs Classe :")
display(pd.crosstab(processed_df["tier"], processed_df["label_name"]))

eda_fig = plot_eda_distributions(processed_df, save_dir=paths["figures"], class_names=class_names)
ipy_display(Image(filename=str(eda_fig)))

## 5. Split Stratifié Groupé (Anti-Leakage), Vocabulaire & Export

Pour éviter toute fuite d'information (*data leakage*), tous les paragraphes issus d'un même article Wikipédia (`title`) sont affectés au **même split** (`train` 70%, `val` 15%, `test` 15%) tout en préservant l'équilibre des classes. Le vocabulaire PyTorch est ensuite construit **uniquement** sur `train`.

In [ ]:
import json
from src.dataset import Vocabulary
from src.preprocessing import stratified_group_split

ratios = cfg["preprocessing"]["split_ratios"]
text_col = cfg["preprocessing"]["representation_for_models"]

train_df, val_df, test_df = stratified_group_split(
    processed_df,
    label_col="label",
    group_col="title",
    train_ratio=ratios["train"],
    val_ratio=ratios["val"],
    seed=cfg["project"]["seed"],
)

# Verification stricte d'absence de fuite entre articles
train_titles = set(train_df["title"])
val_titles = set(val_df["title"])
test_titles = set(test_df["title"])
assert train_titles.isdisjoint(val_titles), "Leakage detecte entre train et val!"
assert train_titles.isdisjoint(test_titles), "Leakage detecte entre train et test!"
assert val_titles.isdisjoint(test_titles), "Leakage detecte entre val et test!"

vocab = Vocabulary().fit(
    train_df[text_col].astype(str).tolist(),
    min_freq=cfg["preprocessing"]["vocab_min_freq"],
    max_size=cfg["preprocessing"]["vocab_max_size"],
)

train_df.to_csv(paths["data_processed"] / "train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(paths["data_processed"] / "val.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(paths["data_processed"] / "test.csv", index=False, encoding="utf-8-sig")
vocab.save(paths["data_processed"] / "vocab.json")

# Export du corpus arabe naturel (paragraphes IA + connexes) pour le fine-tuning AraGPT2 (Partie 2)
for split_name, split_df in (("train", train_df), ("val", val_df), ("test", test_df)):
    corpus_lines = split_df["text"].astype(str).tolist()
    (paths["data_processed"] / f"gpt2_corpus_{split_name}.txt").write_text(
        "\n".join(corpus_lines), encoding="utf-8"
    )
(paths["data_processed"] / "gpt2_corpus.txt").write_text(
    "\n".join(processed_df["text"].astype(str).tolist()), encoding="utf-8"
)

summary = {
    "total_rows": len(processed_df),
    "unique_articles": int(processed_df["title"].nunique()),
    "splits": {
        "train": {"rows": int(len(train_df)), "articles": int(len(train_titles)), "class_dist": {int(k): int(v) for k, v in train_df["label"].value_counts().items()}},
        "val": {"rows": int(len(val_df)), "articles": int(len(val_titles)), "class_dist": {int(k): int(v) for k, v in val_df["label"].value_counts().items()}},
        "test": {"rows": int(len(test_df)), "articles": int(len(test_titles)), "class_dist": {int(k): int(v) for k, v in test_df["label"].value_counts().items()}},
    },
    "vocab_size": len(vocab),
    "preprocessing_stats": prep_stats,
}
(paths["reports"] / "preprocessing_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Export termine avec succes !")
print(f"Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)} | Vocab={len(vocab)}")